# UAV Fault Early-Warning Model — v2 (Rolling Features + Scaler Comparison)

Improvements over the previous version:
- Excludes 2 "continuation" fault onsets (fault already active at flight start —
  no real precursor period exists for these)
- Adds rolling trend/derivative features (short-term volatility, rate of change,
  deviation from each flight's own baseline) for the 6 signals that showed the
  most promise in earlier KDE analysis
- Compares StandardScaler vs RobustScaler, and old vs new features, via proper
  leave-one-fault-flight-out cross-validation — picks the winning combination
  automatically rather than guessing
- Confirmed via CV: natural `scale_pos_weight` (1.0x) already outperforms
  higher multipliers, so we keep it as-is


## 1. Imports

In [100]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    classification_report, precision_recall_curve, auc, fbeta_score,average_precision_score,confusion_matrix,
    accuracy_score, precision_score, recall_score,f1_score,average_precision_score
)

from xgboost import XGBClassifier

pd.set_option("display.width", 120)


## 2. Load Raw Data

Edit the two paths below to point at your raw train and test CSV files.


In [68]:
TRAIN_RAW_PATH = "../data/train_raw.csv"   
TEST_RAW_PATH = "../data/test_raw.csv"     

train_raw = pd.read_csv(TRAIN_RAW_PATH)
test_raw = pd.read_csv(TEST_RAW_PATH)

print(f"Train raw: {train_raw.shape}")
print(f"Test raw:  {test_raw.shape}")


Train raw: (132937, 24)
Test raw:  (47875, 24)


## 3. Preprocessing Functions

In [69]:
FAULT_COLS = ["engine_fault", "aileron_fault", "elevator_fault", "rudder_fault"]
FILL_COLS = [
    "vel_x", "vel_y", "vel_z", "pitch_cmd", "pitch_meas", "pitch_error",
    "yaw_cmd", "yaw_meas", "yaw_error"
]
GAP_NS_THRESHOLD = 1000 * 1e9
HORIZON_SEC = 5.0


def add_flight_id(data):
    data = data.sort_values(by=["bag", "timestamp_ns"]).reset_index(drop=True)
    data["time_gap_ns"] = data.groupby("bag", sort=False)["timestamp_ns"].diff()
    bag_changed = data["bag"] != data["bag"].shift(1)
    large_gap = data["time_gap_ns"] > GAP_NS_THRESHOLD
    data["flight_id"] = (bag_changed | large_gap).cumsum() - 1
    data.drop(columns=["time_gap_ns"], inplace=True)
    return data


def add_flight_time(data):
    data["flight_time_sec"] = data.groupby("flight_id", sort=False)["timestamp_ns"].transform(
        lambda x: (x - x.min()) / 1e9
    )
    return data


def fill_missing(data: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in FILL_COLS if c in data.columns]
    data[cols] = (
        data.groupby("flight_id", sort=False)[cols]
        .ffill()
    )

    return data


def add_engineered_features(data):
    data["yaw_error_clean"] = ((data["yaw_error"] + 180) % 360) - 180
    data["ground_speed"] = np.sqrt(data["vel_x"] ** 2 + data["vel_y"] ** 2 + data["vel_z"] ** 2)
    return data


def add_binary_fault(data):
    present = [c for c in FAULT_COLS if c in data.columns]
    if not present:
        raise KeyError(f"None of {FAULT_COLS} found in the data.")
    data["binary_fault"] = (data[present] > 0).any(axis=1).astype(int)
    return data


def compute_fault_onset(data):
    data = data.sort_values(["flight_id", "flight_time_sec"]).reset_index(drop=True)
    data["fault_onset"] = (
        (data["binary_fault"] == 1)
        & (data.groupby("flight_id")["binary_fault"].shift(1).fillna(0).eq(0))
    ).astype(int)
    return data


def add_future_fault_label(data, horizon_sec=HORIZON_SEC, min_onset_time=1.0):
    """fault_within_5s = 1 if a valid fault_onset occurs within the next
    horizon_sec seconds, within the same flight. Onsets before min_onset_time
    are excluded (fault already active at flight start = no real precursor
    period to learn from)."""
    data = data.sort_values(["flight_id", "flight_time_sec"]).reset_index(drop=True)
    labels = np.zeros(len(data), dtype=int)
    for flight_id, group in data.groupby("flight_id"):
        idx = group.index.values
        times = group["flight_time_sec"].values
        valid_onset = (group["fault_onset"].values == 1) & (times >= min_onset_time)
        onset_times = np.sort(times[valid_onset])
        if len(onset_times) == 0:
            continue
        lo = np.searchsorted(onset_times, times, side="right")
        hi = np.searchsorted(onset_times, times + horizon_sec, side="right")
        labels[idx] = (hi > lo).astype(int)
    data["fault_within_5s"] = labels
    return data


def preprocess(data):
    """Per-file pipeline, up through fault_onset. fault_within_5s is computed
    separately AFTER combining train+test, so onset filtering and horizon
    labeling happen consistently across the full combined pool."""
    data = add_flight_id(data)
    data = add_flight_time(data)
    data = fill_missing(data)
    data = add_engineered_features(data)
    data = add_binary_fault(data)
    data = compute_fault_onset(data)
    data = add_future_fault_label(data)
    return data


FEATURES = [
    "airspeed_cmd", "airspeed_meas", "airspeed_error",
    "roll_cmd", "roll_meas", "roll_error",
    "pitch_cmd", "pitch_meas", "pitch_error",
    "yaw_cmd", "yaw_meas", "yaw_error_clean",
    "vel_x", "vel_y", "vel_z",
    "imu_x", "imu_y", "imu_z",
    "ground_speed",
]

## 4. Apply Preprocessing, Combine Into One Pool

In [70]:
df_train_full = preprocess(train_raw.copy())
df_test_full = preprocess(test_raw.copy())

df_test_offset = df_test_full.copy()
flight_id_offset = df_train_full["flight_id"].max() + 1
df_test_offset["flight_id"] = df_test_offset["flight_id"] + flight_id_offset

df_all = pd.concat([df_train_full, df_test_offset], ignore_index=True)
print(f"Combined: {df_all['flight_id'].nunique()} flights, {len(df_all)} frames")
print(f"Distinct bags: {df_all['bag'].nunique()}  (should match flight count if no over-segmentation)")


Combined: 36 flights, 180812 frames
Distinct bags: 36  (should match flight count if no over-segmentation)


In [82]:
df_xgb_long= (
    df_all
    .sort_values(["flight_id", "flight_time_sec"])
    .reset_index(drop=True)
)

print("Data shape:", df_xgb_long.shape)
print("Flights:", df_xgb_long["flight_id"].nunique())

print("\nTarget distribution:")
print(df_xgb_long["fault_within_5s"].value_counts())

Data shape: (180812, 31)
Flights: 36

Target distribution:
fault_within_5s
0    177310
1      3502
Name: count, dtype: int64


In [83]:
# ============================================================
# LONG CAUSAL HISTORY
# t, t-1, t-2, t-5, t-10, t-20, t-40, t-60
# ============================================================

LAGS = [1, 2, 5, 10, 20, 40, 60]
# ------------------------------------------------------------
# Create lag features
# ------------------------------------------------------------

lagged_data = []

for col in FEATURES:

    # Current t
    lagged_data.append(
        df_xgb_long[col].rename(col)
    )

    # Historical values
    for lag in LAGS:

        lagged_data.append(
            df_xgb_long
            .groupby("flight_id", sort=False)[col]
            .shift(lag)
            .rename(f"{col}_t-{lag}")
        )

# Combine all features at once
df_lag_features = pd.concat(lagged_data, axis=1)

# Replace original feature columns with the generated feature matrix
df_xgb_long = pd.concat(
    [
        df_xgb_long.drop(columns=FEATURES),
        df_lag_features
    ],
    axis=1
)

print("df_xgb_long shape before history removal:", df_xgb_long.shape)

# ------------------------------------------------------------
# Define model feature list
# ------------------------------------------------------------

LAG_FEATURES = []

for col in FEATURES:

    LAG_FEATURES.append(col)

    for lag in LAGS:
        LAG_FEATURES.append(f"{col}_t-{lag}")

print("Number of model features:", len(LAG_FEATURES))

# ------------------------------------------------------------
# Check that every expected feature exists
# ------------------------------------------------------------

missing = [
    col for col in LAG_FEATURES
    if col not in df_xgb_long.columns
]

print("Missing features:", len(missing))

if missing:
    print(missing[:20])

# ------------------------------------------------------------
# Remove first 60 frames of every flight
# ------------------------------------------------------------

history_count = (
    df_xgb_long
    .groupby("flight_id", sort=False)
    .cumcount()
)

df_xgb_long = (
    df_xgb_long.loc[history_count >= max(LAGS)]
    .reset_index(drop=True)
)

print(
    "Shape after removing incomplete history:",
    df_xgb_long.shape
)

print(
    "Number of flights:",
    df_xgb_long["flight_id"].nunique()
)

df_xgb_long shape before history removal: (180812, 164)
Number of model features: 152
Missing features: 0
Shape after removing incomplete history: (178652, 164)
Number of flights: 36


In [84]:
affected_flights = [0, 9, 15, 18, 20, 21, 31, 33]

for fid in affected_flights:
    temp = df_all[df_all["flight_id"] == fid].copy()

    print(
        f"Flight {fid}: "
        f"frames={len(temp)}, "
        f"vel_x NaN={temp['vel_x'].isna().sum()}, "
        f"first NaN index={temp['vel_x'].isna().idxmax() if temp['vel_x'].isna().any() else 'none'}"
    )

    print(
        temp[
            ["flight_time_sec", "vel_x", "vel_y", "vel_z"]
        ].head(10)
    )
    print("-" * 70)

Flight 0: frames=7760, vel_x NaN=9, first NaN index=0
   flight_time_sec     vel_x     vel_y     vel_z
0         0.000000       NaN       NaN       NaN
1         0.000265       NaN       NaN       NaN
2         0.000741       NaN       NaN       NaN
3         0.030340       NaN       NaN       NaN
4         0.071352       NaN       NaN       NaN
5         0.159289       NaN       NaN       NaN
6         0.193384       NaN       NaN       NaN
7         0.241772       NaN       NaN       NaN
8         0.275060       NaN       NaN       NaN
9         0.342245  0.048674 -0.223828  0.007839
----------------------------------------------------------------------
Flight 9: frames=3603, vel_x NaN=6, first NaN index=37657
       flight_time_sec     vel_x     vel_y     vel_z
37657         0.000000       NaN       NaN       NaN
37658         0.000062       NaN       NaN       NaN
37659         0.024857       NaN       NaN       NaN
37660         0.114106       NaN       NaN       NaN
37661        

In [78]:
all_faulty_flights = sorted(
    df_all.loc[df_all["fault_onset"] == 1, "flight_id"].unique()
)

all_flights = sorted(df_all["flight_id"].unique())

non_faulty_flights = sorted(
    set(all_flights) - set(all_faulty_flights)
)

print("Total flights:", len(all_flights))
print("Faulty flights:", len(all_faulty_flights))
print("Normal flights:", len(non_faulty_flights))

Total flights: 36
Faulty flights: 29
Normal flights: 7


In [85]:
np.random.seed(42)

def split_list(lst, train_frac=0.6, val_frac=0.2):
    lst = list(lst)
    np.random.shuffle(lst)

    n = len(lst)
    n_train = int(round(n * train_frac))
    n_val = int(round(n * val_frac))

    return (
        lst[:n_train],
        lst[n_train:n_train + n_val],
        lst[n_train + n_val:]
    )

faulty_train, faulty_val, faulty_test = split_list(all_faulty_flights)
normal_train, normal_val, normal_test = split_list(non_faulty_flights)

train_flights = faulty_train + normal_train
val_flights = faulty_val + normal_val
test_flights = faulty_test + normal_test

print("Train:", len(train_flights))
print("Validation:", len(val_flights))
print("Test:", len(test_flights))

Train: 21
Validation: 7
Test: 8


In [86]:
train_mask_long = df_xgb_long["flight_id"].isin(train_flights)
val_mask_long = df_xgb_long["flight_id"].isin(val_flights)
test_mask_long = df_xgb_long["flight_id"].isin(test_flights)

X_train_long = df_xgb_long.loc[train_mask_long, LAG_FEATURES].copy()
X_val_long = df_xgb_long.loc[val_mask_long, LAG_FEATURES].copy()
X_test_long = df_xgb_long.loc[test_mask_long, LAG_FEATURES].copy()
y_train_long = df_xgb_long.loc[train_mask_long, "fault_within_5s"].copy()
y_val_long = df_xgb_long.loc[val_mask_long, "fault_within_5s"].copy()
y_test_long = df_xgb_long.loc[test_mask_long, "fault_within_5s"].copy()
print("X_train:", X_train_long.shape)
print("X_val:", X_val_long.shape)
print("X_test:", X_test_long.shape)

X_train: (108007, 152)
X_val: (26754, 152)
X_test: (43891, 152)


In [89]:
train_medians_long = X_train_long.median()

X_train_long = X_train_long.fillna(train_medians_long)
X_val_long = X_val_long.fillna(train_medians_long)
X_test_long = X_test_long.fillna(train_medians_long)

print("Train NaNs:", X_train_long.isna().sum().sum())
print("Val NaNs:", X_val_long.isna().sum().sum())
print("Test NaNs:", X_test_long.isna().sum().sum())

Train NaNs: 0
Val NaNs: 0
Test NaNs: 0


In [88]:
neg = (y_train_long == 0).sum()
pos = (y_train_long == 1).sum()

pos_weight = neg / pos

print("Negative:", neg)
print("Positive:", pos)
print("scale_pos_weight:", pos_weight)

Negative: 105742
Positive: 2265
scale_pos_weight: 46.685209713024285


In [90]:
model_long = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=pos_weight,
    eval_metric="aucpr",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

model_long.fit(
    X_train_long,
    y_train_long,
    eval_set=[(X_val_long, y_val_long)],
    verbose=False
)

print("Best iteration:", model_long.best_iteration)
print("Best validation score:", model_long.best_score)

Best iteration: 79
Best validation score: 0.16582800382974988


In [91]:
val_prob_long = model_long.predict_proba(X_val_long)[:, 1]
pr_auc_long = average_precision_score(
    y_val_long,
    val_prob_long
)

print("Validation PR-AUC:", pr_auc_long)

Validation PR-AUC: 0.16936307092919983


In [93]:
val_results_long = df_xgb_long.loc[
    val_mask_long,
    ["flight_id", "flight_time_sec", "fault_within_5s"]
].copy()

val_results_long["probability"] = val_prob_long

val_onsets_long = (
    df_all[
        (df_all["flight_id"].isin(val_flights)) &
        (df_all["fault_onset"] == 1) &
        (df_all["flight_time_sec"] >= 5.0)
    ][
        ["flight_id", "flight_time_sec"]
    ]
    .drop_duplicates()
    .sort_values(["flight_id", "flight_time_sec"])
)

print("Validation fault events:", len(val_onsets_long))
print(val_onsets_long)

Validation fault events: 7
        flight_id  flight_time_sec
16762           3       196.069189
108596         22       569.134332
111363         22       713.091892
125939         27       122.531777
128435         27       728.590486
131148         28       118.833552
180651         35       123.206626


In [95]:
def evaluate_event_level(
    val_results,
    val_onsets,
    threshold,
    val_flights
):

    results = val_results.copy()

    results["prediction"] = (
        results["probability"] >= threshold
    ).astype(int)

    event_results = []

    # ============================================================
    # 1. EVENT DETECTION
    # ============================================================

    for _, event in val_onsets.iterrows():

        flight_id = event["flight_id"]
        onset_time = event["flight_time_sec"]

        # Strictly before the fault onset
        # and within the previous 5 seconds
        window = results[
            (results["flight_id"] == flight_id) &
            (results["flight_time_sec"] > onset_time - 5.0) &
            (results["flight_time_sec"] < onset_time)
        ]

        detected = window["prediction"].any()

        if detected:

            detected_rows = window[
                window["prediction"] == 1
            ]

            first_warning_time = detected_rows[
                "flight_time_sec"
            ].min()

            warning_lead_time = (
                onset_time - first_warning_time
            )

        else:

            first_warning_time = np.nan
            warning_lead_time = np.nan

        event_results.append({
            "flight_id": flight_id,
            "onset_time": onset_time,
            "detected": detected,
            "first_warning_time": first_warning_time,
            "warning_lead_sec": warning_lead_time
        })

    event_results = pd.DataFrame(event_results)

    event_recall = event_results["detected"].mean()

    # ============================================================
    # 2. FALSE-ALARM EPISODES
    # ============================================================

    false_alarm_count = 0

    for flight_id, group in results.groupby("flight_id"):

        group = group.sort_values(
            "flight_time_sec"
        ).copy()

        # Start of a predicted-positive episode
        group["positive_start"] = (
            (group["prediction"] == 1) &
            (group["prediction"].shift(1).fillna(0) == 0)
        )

        start_indices = group.index[
            group["positive_start"]
        ]

        for start_idx in start_indices:

            start_pos = group.index.get_loc(start_idx)

            segment = group.iloc[start_pos:]

            # End episode at first predicted-negative frame
            negative_positions = np.where(
                segment["prediction"].values == 0
            )[0]

            if len(negative_positions) > 0:

                end_pos = negative_positions[0] - 1

                segment = segment.iloc[:end_pos + 1]

            # If this positive episode contains no
            # fault-within-5s frames, count as false alarm
            if not segment["fault_within_5s"].eq(1).any():

                false_alarm_count += 1

    false_alarms_per_flight = (
        false_alarm_count / len(val_flights)
    )

    # ============================================================
    # 3. RETURN SUMMARY
    # ============================================================

    return {
        "threshold": threshold,
        "event_recall": event_recall,
        "detected_events": int(
            event_results["detected"].sum()
        ),
        "total_events": len(event_results),
        "false_alarm_events": false_alarm_count,
        "false_alarms_per_flight": false_alarms_per_flight,
        "mean_warning_lead_sec": (
            event_results.loc[
                event_results["detected"],
                "warning_lead_sec"
            ].mean()
        )
    }, event_results

In [96]:
thresholds = np.arange(0.10, 0.91, 0.05)

results_long = []

for threshold in thresholds:

    result, event_results = evaluate_event_level(
        val_results=val_results_long,
        val_onsets=val_onsets_long,
        threshold=threshold,
        val_flights=val_flights
    )

    results_long.append(result)

threshold_df_long = pd.DataFrame(results_long)

threshold_df_long

,threshold,event_recall,detected_events,total_events,false_alarm_events,false_alarms_per_flight,mean_warning_lead_sec
0,0.10,1.000000,7,7,149,21.285714,4.978879
1,0.15,1.000000,7,7,134,19.142857,4.978879
2,0.20,0.857143,6,7,120,17.142857,4.941952
3,0.25,0.857143,6,7,124,17.714286,4.941952
4,0.30,0.857143,6,7,102,14.571429,4.941952
5,0.35,0.857143,6,7,96,13.714286,4.931060
6,0.40,0.857143,6,7,75,10.714286,4.931060
7,0.45,0.857143,6,7,78,11.142857,4.820409
8,0.50,0.857143,6,7,66,9.428571,4.758782
9,0.55,0.857143,6,7,59,8.428571,4.429280


In [97]:
result_60_long, event_results_60_long = evaluate_event_level(
    val_results=val_results_long,
    val_onsets=val_onsets_long,
    threshold=0.60,
    val_flights=val_flights
)

event_results_60_long

,flight_id,onset_time,detected,first_warning_time,warning_lead_sec
0,3.0,196.069189,True,192.134930,3.934259
1,22.0,569.134332,True,564.310594,4.823737
2,22.0,713.091892,True,708.115945,4.975947
3,27.0,122.531777,False,NaN,NaN
4,27.0,728.590486,True,723.628333,4.962153
5,28.0,118.833552,True,114.055037,4.778515
6,35.0,123.206626,True,120.302504,2.904121


In [98]:
# Flight 27, first fault event
flight_id = 27
onset_time = 122.531777

missed_event = val_results_long[
    (val_results_long["flight_id"] == flight_id) &
    (val_results_long["flight_time_sec"] > onset_time - 5.0) &
    (val_results_long["flight_time_sec"] < onset_time)
].copy()

print(
    missed_event[
        ["flight_id", "flight_time_sec", "probability"]
    ].to_string(index=False)
)

print("\nMaximum probability:")
max_idx = missed_event["probability"].idxmax()

print("Probability:", missed_event.loc[max_idx, "probability"])
print("Time:", missed_event.loc[max_idx, "flight_time_sec"])
print(
    "Lead time:",
    onset_time - missed_event.loc[max_idx, "flight_time_sec"]
)

 flight_id  flight_time_sec  probability
        27       117.532741     0.159358
        27       117.557885     0.159358
        27       117.606844     0.148044
        27       117.661629     0.146856
        27       117.685504     0.137807
        27       117.778165     0.122889
        27       117.805598     0.120240
        27       117.880619     0.120240
        27       117.892666     0.118134
        27       117.955816     0.125209
        27       118.003555     0.082095
        27       118.059283     0.098241
        27       118.096303     0.098241
        27       118.127582     0.091017
        27       118.209396     0.093471
        27       118.231859     0.071989
        27       118.312444     0.064877
        27       118.343100     0.069099
        27       118.413079     0.071850
        27       118.449337     0.069393
        27       118.509236     0.057831
        27       118.535966     0.057831
        27       118.629863     0.057831
        27      

In [113]:
threshold = 0.60

y_val_pred_long = (
    val_prob_long >= threshold
).astype(int)

print("=" * 60)
print("LONG-HISTORY XGBOOST — VALIDATION RESULTS")
print("=" * 60)

print(f"Threshold: {threshold}")
print(f"PR-AUC: {average_precision_score(y_val_long, val_prob_long):.4f}")
print(f"Accuracy: {accuracy_score(y_val_long, y_val_pred_long):.4f}")
print(f"Precision: {precision_score(y_val_long, y_val_pred_long):.4f}")
print(f"Recall: {recall_score(y_val_long, y_val_pred_long):.4f}")
print(f"F1: {f1_score(y_val_long, y_val_pred_long):.4f}")
print(f"F0.5: {fbeta_score(y_val_long, y_val_pred_long, beta=0.5):.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_val_long,
        y_val_pred_long,
        target_names=["Healthy (0)", "Fault within 5s (1)"],
        digits=4
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_long, y_val_pred_long))

LONG-HISTORY XGBOOST — VALIDATION RESULTS
Threshold: 0.6
PR-AUC: 0.1694
Accuracy: 0.9374
Precision: 0.1867
Recall: 0.4491
F1: 0.2637
F0.5: 0.2114

Classification Report:
                     precision    recall  f1-score   support

        Healthy (0)     0.9854    0.9499    0.9673     26086
Fault within 5s (1)     0.1867    0.4491    0.2637       668

           accuracy                         0.9374     26754
          macro avg     0.5860    0.6995    0.6155     26754
       weighted avg     0.9654    0.9374    0.9497     26754


Confusion Matrix:
[[24779  1307]
 [  368   300]]


In [114]:
threshold = 0.70

y_val_pred_long_070 = (
    val_prob_long >= threshold
).astype(int)

print("=" * 60)
print("LONG-HISTORY XGBOOST — THRESHOLD 0.70")
print("=" * 60)

print(f"PR-AUC:   {average_precision_score(y_val_long, val_prob_long):.4f}")
print(f"Accuracy: {accuracy_score(y_val_long, y_val_pred_long_070):.4f}")
print(f"Precision:{precision_score(y_val_long, y_val_pred_long_070):.4f}")
print(f"Recall:   {recall_score(y_val_long, y_val_pred_long_070):.4f}")
print(f"F1:       {f1_score(y_val_long, y_val_pred_long_070):.4f}")
print(f"F0.5:       {fbeta_score(y_val_long, y_val_pred_long_070, beta=0.5):.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_val_long,
        y_val_pred_long_070,
        target_names=["Healthy (0)", "Fault within 5s (1)"],
        digits=4
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_long, y_val_pred_long_070))

LONG-HISTORY XGBOOST — THRESHOLD 0.70
PR-AUC:   0.1694
Accuracy: 0.9466
Precision:0.1849
Recall:   0.3338
F1:       0.2380
F0.5:       0.2030

Classification Report:
                     precision    recall  f1-score   support

        Healthy (0)     0.9826    0.9623    0.9723     26086
Fault within 5s (1)     0.1849    0.3338    0.2380       668

           accuracy                         0.9466     26754
          macro avg     0.5837    0.6481    0.6052     26754
       weighted avg     0.9627    0.9466    0.9540     26754


Confusion Matrix:
[[25103   983]
 [  445   223]]


In [116]:
threshold_results = []

for threshold in np.arange(0.05, 0.91, 0.01):

    y_pred = (val_prob_long >= threshold).astype(int)

    f2 = fbeta_score(
        y_val_long,
        y_pred,
        beta=0.5
    )

    precision = precision_score(
        y_val_long,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val_long,
        y_pred,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_val_long,
        y_pred
    )

    event_metrics, _ = evaluate_event_level(
        val_results_long,
        val_onsets_long,
        threshold,
        val_flights
    )

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f0.5": f2,
        "event_recall": event_metrics["event_recall"],
        "false_alarms": event_metrics["false_alarm_events"],
        "false_alarms_per_flight": event_metrics["false_alarms_per_flight"],
        "mean_warning_lead_sec": event_metrics["mean_warning_lead_sec"]
    })

threshold_df = pd.DataFrame(threshold_results)

# Best F2
best_f2 = threshold_df.loc[
    threshold_df["f0.5"].idxmax()
]

print("BEST F0.5")
print(best_f2.to_string())

BEST F0.5
threshold                   0.840000
accuracy                    0.965015
precision                   0.266551
recall                      0.229042
f0.5                        0.258097
event_recall                0.428571
false_alarms               32.000000
false_alarms_per_flight     4.571429
mean_warning_lead_sec       3.548153


In [118]:
# Thresholds that maintain at least 80% event recall
candidates = threshold_df[
    threshold_df["event_recall"] >= 0.80
].copy()

# Among them, choose the highest threshold
best_constrained = candidates.loc[
    candidates["threshold"].idxmax()
]

print("=" * 60)
print("BEST PRECISION-FOCUSED THRESHOLD")
print("(Event Recall >= 80%)")
print("=" * 60)
print(best_constrained.to_string())

BEST PRECISION-FOCUSED THRESHOLD
(Event Recall >= 80%)
threshold                   0.700000
accuracy                    0.946625
precision                   0.184909
recall                      0.333832
f0.5                        0.203023
event_recall                0.857143
false_alarms               49.000000
false_alarms_per_flight     7.000000
mean_warning_lead_sec       3.564842


In [119]:
FINAL_THRESHOLD = 0.70

In [122]:
test_prob_long = model_long.predict_proba(X_test_long)[:, 1]
# --------------------------------------------------
# Frame-level test evaluation
# --------------------------------------------------

y_test_pred = (
    test_prob_long >= FINAL_THRESHOLD
).astype(int)

test_precision = precision_score(
    y_test_long,
    y_test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test_long,
    y_test_pred,
    zero_division=0
)

test_f05 = fbeta_score(
    y_test_long,
    y_test_pred,
    beta=0.5
)

test_accuracy = accuracy_score(
    y_test_long,
    y_test_pred
)

test_pr_auc = average_precision_score(
    y_test_long,
    test_prob_long
)

print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)
print(f"Threshold : {FINAL_THRESHOLD:.2f}")
print(f"PR-AUC    : {test_pr_auc:.4f}")
print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F0.5      : {test_f05:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_long, y_test_pred))

FINAL TEST RESULTS
Threshold : 0.70
PR-AUC    : 0.0800
Accuracy  : 0.9778
Precision : 0.0825
Recall    : 0.0703
F0.5      : 0.0797

Confusion Matrix:
[[42877   445]
 [  529    40]]


In [123]:
test_event_metrics, test_event_results = evaluate_event_level(
    test_results_long,
    test_onsets_long,
    FINAL_THRESHOLD,
    test_flights
)

print("=" * 60)
print("FINAL TEST — EVENT LEVEL")
print("=" * 60)

for key, value in test_event_metrics.items():
    print(f"{key}: {value}")

print("\nIndividual Events:")
print(test_event_results.to_string(index=False))

FINAL TEST — EVENT LEVEL
threshold: 0.7
event_recall: 0.16666666666666666
detected_events: 1
total_events: 6
false_alarm_events: 30
false_alarms_per_flight: 3.75
mean_warning_lead_sec: 3.3802184319999924

Individual Events:
 flight_id  onset_time  detected  first_warning_time  warning_lead_sec
       7.0  132.635650     False                 NaN               NaN
       8.0  164.152501     False                 NaN               NaN
      11.0  475.754518      True            472.3743          3.380218
      17.0  154.043161     False                 NaN               NaN
      23.0  666.479451     False                 NaN               NaN
      24.0   83.573811     False                 NaN               NaN
